# Proyecto: Impacto de la Inteligencia Artificial en el Empleo (Hacia 2030)

## Problemática

La rápida adopción de la Inteligencia Artificial (IA) está transformando los entornos laborales. Si bien optimiza la productividad y la eficiencia, también genera una profunda incertidumbre sobre el futuro del empleo y la automatización de funciones.

El desafío principal radica en que **el impacto no es uniforme**: varía significativamente según el perfil de cada ocupación.

## Pregunta orientadora

¿Cómo podría impactar la adopción de la Inteligencia Artificial en las distintas ocupaciones laborales hacia el año 2030 de acuerdo con las variables contenidas en el dataset seleccionado?

## El Desafío Analítico

La problemática de este proyecto consiste en **identificar y analizar qué factores clave determinan el nivel de riesgo de transformación laboral** frente a la IA hacia el año 2030.

Para resolverlo, utilizaremos el conjunto de datos "AI Impact on Jobs 2030" con el fin de evaluar cómo interactúan las siguientes variables:

- **Exposición tecnológica** y probabilidad de automatización.
- **Nivel educacional** requerido.
- **Experiencia laboral** previa.

**Objetivo del Notebook:** Construir y ejecutar el pipeline completo de preprocesamiento para dejar el dataset listo para modelado.

### Información de atributos del dataset

**1. Variables principales**
- `Job_Title` — Categórica nominal. Nombre de la ocupación (20 categorías).
- `Average_Salary` — Numérica (int). Salario promedio anual.
- `Years_Experience` — Numérica (int). Años de experiencia laboral.
- `Education_Level` — Categórica ordinal. Nivel educativo (High School, Bachelor's, Master's, PhD).

**2. Variables relacionadas con IA y automatización**
- `AI_Exposure_Index` — Numérica float [0–1]. Nivel de exposición del trabajo a la IA.
- `Tech_Growth_Factor` — Numérica float. Tasa de crecimiento tecnológico del sector.
- `Automation_Probability_2030` — Numérica float [0–1]. Probabilidad estimada de automatización hacia 2030.
- `Risk_Category` — Categórica ordinal. Nivel de riesgo: Low, Medium, High.

**3. Variables de habilidades (skills)**
- `Skill_1` a `Skill_10` — Numéricas float [0–1]. Niveles de dominio en distintas habilidades (normalizadas en origen).

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt

# Agregar src al path de Python
sys.path.append(os.path.abspath("../src"))

from data_loading import cargar_datos
from preprocessing import (
    limpiar_datos,
    encoding_categorico,
    crear_features,
    normalizar_datos,
    validar_datos,
    exportar_dataset
)

print("Módulos cargados correctamente.")

---

## 1. Carga y Exploración Inicial

Se carga el dataset y se realiza un diagnóstico inicial antes de aplicar cualquier transformación: dimensiones, tipos de datos, valores nulos y duplicados.

In [ ]:
ds = cargar_datos('../data/raw/AI_Impact_on_Jobs_2030.csv')

print("=== DIAGNÓSTICO INICIAL ===")
print(f"\nDimensiones: {ds.shape[0]} filas × {ds.shape[1]} columnas")
print("\n--- Información general ---")
ds.info()

In [ ]:
print("--- Estadísticas descriptivas ---")
display(ds.describe())

print("\n--- Valores nulos por columna ---")
print(ds.isna().sum())

print(f"\n--- Registros duplicados: {ds.duplicated().sum()} ---")

print("\n--- Primeras 5 filas ---")
display(ds.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ds['Risk_Category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribución de Risk_Category')
axes[0].set_xlabel('Categoría de riesgo')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=0)

ds['Education_Level'].value_counts().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Distribución de Education_Level')
axes[1].set_xlabel('Nivel educativo')
axes[1].set_ylabel('Frecuencia')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

---

## 2. Limpieza de Datos

Se aplica `limpiar_datos()` del módulo `src/preprocessing.py`. Esta función:

- Elimina registros duplicados con `drop_duplicates()`.
- Imputa valores nulos numéricos con la **mediana** de cada columna (método robusto ante outliers).
- Imputa valores nulos categóricos con la **moda** de cada columna.

Se muestran los conteos antes y después para evidenciar el efecto de la limpieza.

In [ ]:
nulos_antes = ds.isna().sum().sum()
duplicados_antes = ds.duplicated().sum()
filas_antes = len(ds)

print(f"Antes de limpiar — nulos: {nulos_antes}, duplicados: {duplicados_antes}, filas: {filas_antes}")

ds = limpiar_datos(ds)

nulos_despues = ds.isna().sum().sum()
duplicados_despues = ds.duplicated().sum()
filas_despues = len(ds)

print(f"Después de limpiar — nulos: {nulos_despues}, duplicados: {duplicados_despues}, filas: {filas_despues}")
print(f"\nFilas eliminadas: {filas_antes - filas_despues}")
print(f"Nulos imputados: {nulos_antes - nulos_despues}")

---

## 3. Transformación y Encoding

Se aplica `encoding_categorico()` del módulo `src/preprocessing.py`. Las decisiones técnicas son:

| Variable | Tipo | Método | Justificación |
|---|---|---|---|
| `Job_Title` | Nominal (sin orden) | One-Hot Encoding (`pd.get_dummies`) | No existe jerarquía entre ocupaciones; OHE evita introducir un orden artificial |
| `Education_Level` | Ordinal (con orden) | Encoding ordinal con mapeo explícito | Existe orden lógico: High School < Bachelor's < Master's < PhD; se preserva la relación jerárquica |
| `Risk_Category` | Ordinal (con orden) | Encoding ordinal con mapeo explícito | Existe orden semántico: Low < Medium < High; preservarlo permite a los modelos explotar esa relación |

In [ ]:
shape_antes = ds.shape
print(f"Shape antes del encoding: {shape_antes}")

ds = encoding_categorico(ds)

print(f"Shape después del encoding: {ds.shape}")
print(f"Columnas nuevas: {ds.shape[1] - shape_antes[1]}")

print(f"\nEducation_Level (ordinal) — valores únicos: {sorted(ds['Education_Level'].unique())}")
print(f"Risk_Category (ordinal) — valores únicos: {sorted(ds['Risk_Category'].unique())}")

job_cols = [c for c in ds.columns if c.startswith('Job_Title_')]
print(f"\nColumnas OHE generadas para Job_Title ({len(job_cols)} categorías):")
print(job_cols)

---

## 4. Feature Engineering

Se aplica `crear_features()` del módulo `src/preprocessing.py`. Esta función genera dos nuevas variables:

- **`Skill_Index`:** promedio de `Skill_1` a `Skill_10`. Captura el perfil general de habilidades con una variable agregada, reduciendo dimensionalidad sin perder información de conjunto.
- **`High_Risk`:** variable objetivo binaria. Valor 1 si `Automation_Probability_2030 > 0.7`, 0 en caso contrario. El umbral del 70% corresponde a la clasificación de riesgo alto adoptada en la literatura sobre automatización laboral.

In [ ]:
ds = crear_features(ds)

print("Nuevas variables creadas:")

print(f"\nHigh_Risk (Automation_Probability_2030 > 0.7):")
print(ds['High_Risk'].value_counts())
print(f"  Porcentaje de alto riesgo: {ds['High_Risk'].mean()*100:.1f}%")

print(f"\nSkill_Index (promedio Skill_1 a Skill_10):")
print(ds['Skill_Index'].describe().round(3))

---

## 5. Normalización

Se aplica `normalizar_datos()` del módulo `src/preprocessing.py`. Las variables con escalas distintas a [0,1] se normalizan con `MinMaxScaler` para garantizar que ninguna variable domine por su magnitud.

**Variables normalizadas:** `Average_Salary`, `Years_Experience`, `Tech_Growth_Factor`, `Education_Level`, `Risk_Category`.

**Variables que ya estaban en [0,1]** (no se tocan): `AI_Exposure_Index`, `Automation_Probability_2030`, `Skill_1`–`Skill_10`, `Skill_Index`, `High_Risk`, columnas `Job_Title_*`.

**Elección de MinMaxScaler sobre StandardScaler:** el dataset no contiene outliers extremos y se quiere mantener los valores en el rango [0,1] para consistencia con las demás variables.

In [ ]:
cols_a_ver = ['Average_Salary', 'Years_Experience', 'Tech_Growth_Factor']

print("Antes de normalizar:")
display(ds[cols_a_ver].describe().round(2))

ds = normalizar_datos(ds)

print("\nDespués de normalizar:")
display(ds[cols_a_ver].describe().round(4))

---

## 6. Validación

Se verifican las condiciones de calidad del dataset final:

- **Casos normales:** ausencia de nulos y duplicados.
- **Casos límite:** rangos esperados para variables clave.
- **Tipos de datos:** variables numéricas con tipo correcto tras normalización.

Cada validación muestra evidencia explícita del resultado.

In [ ]:
print("=== VALIDACIONES DEL DATASET FINAL ===")
print(f"\nDimensiones: {ds.shape[0]} filas × {ds.shape[1]} columnas")
print(f"Nulos totales: {ds.isna().sum().sum()}")
print(f"Duplicados totales: {ds.duplicated().sum()}")

# Validaciones de rango
assert ds['AI_Exposure_Index'].between(0, 1).all(), "❌ AI_Exposure_Index fuera de [0,1]"
print("\n✅ AI_Exposure_Index en rango [0,1]")

assert ds['Automation_Probability_2030'].between(0, 1).all(), "❌ Automation_Probability_2030 fuera de [0,1]"
print("✅ Automation_Probability_2030 en rango [0,1]")

assert ds['High_Risk'].isin([0, 1]).all(), "❌ High_Risk tiene valores distintos de 0 y 1"
print("✅ High_Risk solo contiene 0 y 1")

assert ds['Education_Level'].between(0, 1).all(), "❌ Education_Level fuera de [0,1] tras normalización"
print("✅ Education_Level en rango [0,1] (normalizado)")

assert ds['Risk_Category'].between(0, 1).all(), "❌ Risk_Category fuera de [0,1] tras normalización"
print("✅ Risk_Category en rango [0,1] (normalizado)")

assert ds['Average_Salary'].between(0, 1).all(), "❌ Average_Salary fuera de [0,1] tras normalización"
print("✅ Average_Salary en rango [0,1] (normalizado)")

# Validación de tipos
assert ds['Average_Salary'].dtype in ['float64', 'float32'], "❌ Average_Salary debe ser float"
print("✅ Average_Salary tipo float")

# Validación de columnas OHE
job_cols = [c for c in ds.columns if c.startswith('Job_Title_')]
assert len(job_cols) == 20, f"❌ Se esperaban 20 columnas Job_Title_*, se encontraron {len(job_cols)}"
print(f"✅ Columnas OHE Job_Title_*: {len(job_cols)} (correcto)")

# Validación del módulo
print("")
validar_datos(ds)

---

## 7. Exportación

Se exporta el dataset procesado a `data/processed/AI_Impact_on_Jobs_2030_clean.csv` usando `exportar_dataset()` del módulo `src/preprocessing.py`.

In [ ]:
exportar_dataset(ds)